# Парсинг отчетов 10-К

Скачать файлы с сайта формата 10-К можно несколькими способами. Существуют готовые инструменты для python:


*   edgartools
*   sec-edgar-api




Парсинг HTML-файлов с отчетами 10-К

Итог: файл JSON формата для отчета каждого года каждой компании




> {
  > "company": "fnm",
  > "fiscal_year": 2025,
> "file_name": "fnm-20251231",
> "section_item_1": "...",
> "section_item_1a": "...",
> "section_item_7": "...",
> "section_item_7a": "..."
> }





Конфигурация: указать нужные пути к папкам

In [ ]:
# путь к папке с HTML файлами 10-K отчетов из EDGAR
REPORTS_DIR = "/Users/dariazvereva/Documents/kursovaya/all_texts"

# путь для сохранения JSON файлов с результатами
OUTPUT_DIR = "/Users/dariazvereva/Documents/kursovaya/ВЫБОРКА"

In [ ]:
import os
import re
import json
from bs4 import BeautifulSoup
from pathlib import Path
from typing import Dict, Optional


class TenKParser:
    def __init__(self, html_path: str):
        with open(html_path, 'r', encoding='utf-8') as f:
            content = f.read()
            if b'\\x' in content.encode('utf-8'):
                try:
                    content = content.encode('latin-1').decode('utf-8')
                except:
                    pass
            self.soup = BeautifulSoup(content, 'html.parser')
        self.file_name = Path(html_path).stem

    def extract_item(self, item_num: str) -> Optional[str]:
        """
        Извлекает секцию item_num.
        item_num: '1', '1a', '7', '7a'
        """
        patterns = self._build_patterns(item_num)

        for pattern in patterns:
            text = self._extract_by_toc_link(pattern)
            if text:
                return text

            text = self._extract_by_tag_attributes(pattern)
            if text:
                return text

            text = self._extract_by_header_tag(pattern)
            if text:
                return text

            text = self._extract_by_text_search(pattern)
            if text:
                return text

        return None

    def _build_patterns(self, item_num: str) -> list:
        patterns = []

        base = item_num.lower()
        patterns.extend([
            rf'item\s+{base}\b',
            rf'item\s+{base}\.',
            rf'item\s+{base}\s*:',
            rf'item\s+{base}\s+[-–]',
            rf'^{base}\.',
            rf'^{base}\s+',
        ])

        roman_map = {'1': 'I', '1a': 'IA', '7': 'VII', '7a': 'VIIA'}
        if item_num in roman_map:
            patterns.append(rf'PART\s+{roman_map[item_num]}\b')

        return patterns

    def _extract_by_toc_link(self, pattern) -> Optional[str]:
        for link in self.soup.find_all('a', href=True):
            link_text = link.get_text(strip=True).lower()
            if re.search(pattern, link_text, re.IGNORECASE):
                href = link['href']
                if href.startswith('#'):
                    target_id = href[1:]
                    target = self.soup.find(id=target_id)
                    if target:
                        return self._extract_section_content(target)
        return None

    def _extract_by_tag_attributes(self, pattern) -> Optional[str]:
        for tag in ['a', 'div', 'span', 'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'b', 'strong']:
            elements = self.soup.find_all(tag, {'name': re.compile(pattern, re.IGNORECASE)})
            elements += self.soup.find_all(tag, {'id': re.compile(pattern, re.IGNORECASE)})
            for el in elements:
                text = self._extract_section_content(el)
                if text:
                    return text
        return None

    def _extract_by_header_tag(self, pattern) -> Optional[str]:
        for tag in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'b', 'strong', 'font', 'p']:
            for element in self.soup.find_all(tag):
                text = element.get_text(strip=True)
                if re.search(pattern, text, re.IGNORECASE):
                    if element.find_parent('table'):
                        parent_table = element.find_parent('table')
                        if parent_table and 'toc' in parent_table.get_text(strip=True).lower():
                            continue
                    return self._extract_section_content(element)
        return None

    def _extract_by_text_search(self, pattern) -> Optional[str]:
        elements = self.soup.find_all(string=re.compile(pattern, re.IGNORECASE))
        for elem in elements:
            parent = elem.parent
            if parent:
                if parent.find_parent('table'):
                    continue
                text = self._extract_section_content(parent)
                if text and len(text) > 100:
                    return text
        return None

    def _extract_section_content(self, start_element) -> Optional[str]:
        texts = []
        current = start_element

        if not start_element.name in ['div', 'p', 'section']:
            parent = start_element.find_parent(['div', 'p', 'section', 'td'])
            if parent:
                current = parent

        max_depth = 0

        while current and max_depth < 500:
            max_depth += 1

            if current != start_element:
                if self._is_next_section(current):
                    break

            text = self._clean_text(current)
            if text and len(text) > 30:
                texts.append(text)

            next_elem = current.find_next()
            if not next_elem or next_elem == current:
                break
            current = next_elem

        if texts:
            return '\n\n'.join(texts)
        return None

    def _clean_text(self, element) -> str:
        el_clone = element.__copy__()

        for tag in el_clone.find_all(['ix:nonNumeric', 'ix:nonFraction', 'ix:hidden', 'ix:header']):
            tag.decompose()

        for div in el_clone.find_all('div', style=re.compile(r'display\s*:\s*none', re.IGNORECASE)):
            div.decompose()

        text = el_clone.get_text(strip=True)

        if len(text) < 30:
            return ''

        return text

    def _is_next_section(self, element) -> bool:
        text = element.get_text(strip=True).lower()

        next_patterns = [
            r'item\s+\d+',
            r'part\s+\w+',
            r'^article\s+\d+',
            r'^section\s+\d+',
        ]

        for pattern in next_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                if not re.search(r'item\s+[17]', text, re.IGNORECASE):
                    return True
        return False

    def parse_all(self) -> Dict:
        return {
            'company': self._guess_company_name(),
            'fiscal_year': self._guess_fiscal_year(),
            'file_name': self.file_name,
            'section_item_1': self.extract_item('1'),
            'section_item_1a': self.extract_item('1a'),
            'section_item_7': self.extract_item('7'),
            'section_item_7a': self.extract_item('7a'),
        }

    def _guess_company_name(self) -> str:
        name_tag = self.soup.find('ix:nonNumeric', {'name': 'dei:EntityRegistrantName'})
        if name_tag:
            return name_tag.get_text(strip=True)

        title = self.soup.find('title')
        if title:
            name = re.sub(r'\s*10-K.*$', '', title.text, flags=re.I)
            name = re.sub(r'-\d{8}$', '', name)
            return name.strip()

        for h1 in self.soup.find_all(['h1', 'h2']):
            text = h1.get_text(strip=True)
            if 'INC.' in text.upper() or 'CORP' in text.upper():
                return text[:100]

        return self.file_name.replace('_', ' ').title()

    def _guess_fiscal_year(self) -> Optional[int]:
        year_tag = self.soup.find('ix:nonNumeric', {'name': 'dei:DocumentFiscalYearFocus'})
        if year_tag:
            try:
                return int(year_tag.get_text(strip=True))
            except:
                pass

        text = self.soup.get_text()
        patterns = [
            r'Fiscal Year Ended\s+(\d{4})',
            r'For the fiscal year ended\s+\w+\s+\d{1,2},\s+(\d{4})',
            r'Year Ended\s+\w+\s+\d{1,2},\s+(\d{4})',
            r'January\s+31,\s+(\d{4})',
            r'December\s+31,\s+(\d{4})',
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return int(match.group(1))
        return None


def process_all_reports(reports_dir: str, output_dir: str):
    html_files = list(Path(reports_dir).glob('*.html'))

    if not html_files:
        print(f"нет HTML файлов в {reports_dir}")
        return

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    all_reports = []

    for html_path in html_files:
        try:
            parser = TenKParser(str(html_path))
            data = parser.parse_all()

            output_path = Path(output_dir) / f"{html_path.stem}.json"
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)

            stats = []
            for item in ['section_item_1', 'section_item_1a', 'section_item_7', 'section_item_7a']:
                status = 'да' if data[item] and len(data[item]) > 100 else 'нет'
                stats.append(f"{item.replace('section_', '')}={status}")

            print(f"  -> {' '.join(stats)}")
            if data['section_item_1']:
                print(f"      Item 1: {len(data['section_item_1'])} символов")

        except Exception as e:
            print(f"  -> ОШИБКА: {e}")




if __name__ == "__main__":
    process_all_reports(REPORTS_DIR, OUTPUT_DIR)